# 02 — Recurrence and the Resolution Threshold

## Claim

The thesis claims that:
1. **Poincaré recurrence time** grows combinatorially with nesting depth
2. Inner orbits recur fast; outer orbits recur astronomically slowly
3. This creates a **continuous gradient** from "returnable" (inner) to "non-returnable" (outer)
4. The gradient is real — but parsing it as a binary "spatial vs. temporal" split is the Cartesian descriptive step
5. Coordinate identification (matching position to a returning coordinate) succeeds on inner orbits and fails on outer — a **resolution threshold**

## Model

For $N$ coupled oscillators with incommensurate frequencies (prime ratios), compute:
- Exact recurrence time as a function of $N$
- How "close" the system gets to initial state at various time horizons
- The gradient from fast recurrence (innermost) to astronomical recurrence (outermost)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sympy import lcm, Rational, pi as sym_pi, Integer
import seaborn as sns

sns.set_theme(style='whitegrid')
%matplotlib inline

### Exact Recurrence Times

For oscillators with angular frequencies $\omega_i = 2\pi p_i$ (where $p_i$ are primes),
the period of oscillator $i$ is $T_i = 1/p_i$.

The recurrence time for the first $k$ oscillators is the LCM of their periods.
Since primes are mutually coprime: $T_{\text{rec}}(k) = \text{lcm}(T_1, T_2, \ldots, T_k) = \frac{1}{\gcd(p_1, p_2, \ldots, p_k)} = \frac{\prod p_i}{\text{gcd terms}}$

For integer-frequency oscillators with periods $1/p_i$, the joint recurrence time is $1 / \gcd(p_1, p_2, \ldots)$.
Since primes are coprime, $\gcd = 1$, so joint recurrence = 1 for all subsets.

But this is **exact** recurrence. The thesis concerns **state** recurrence —
the total configuration returning to within $\epsilon$ of initial conditions.
With incommensurate perturbations or nonlinear coupling, exact recurrence becomes approximate,
and the recurrence time depends on the required precision $\epsilon$.

We explore both exact and approximate recurrence.

In [ ]:
# Exact recurrence: periods and LCMs
primes = [2, 3, 5, 7, 11, 13, 17, 19, 23, 29]  # extend beyond 4 to show scaling

# With frequencies f_i = p_i, periods T_i = 1/p_i
# LCM of rationals 1/p_1, 1/p_2, ... = 1/gcd(p_1, p_2, ...)
# For primes, gcd = 1 always. So exact angular recurrence = period 1.
# But the NUMBER OF DISTINCT STATES visited before recurrence = product of primes.

# Think of it differently: discretise each orbit into p_i positions.
# The total state space has prod(p_i) states.
# Recurrence visits ALL states before returning.

state_space_sizes = []
running_product = 1
for p in primes:
    running_product *= p
    state_space_sizes.append(running_product)

print('Nesting depth → Total state space size (product of primes):')
print('=' * 55)
for i, (p, ss) in enumerate(zip(primes, state_space_sizes), 1):
    print(f'  Depth {i} (prime {p:2d}): {ss:>15,} states')

print(f'\nRatio outer/inner for 4 primes (2,3,5,7):')
ss4 = state_space_sizes[3]  # 2*3*5*7 = 210
print(f'  Total: {ss4} states')
print(f'  Inner (just orbit 2): 2 states')
print(f'  Ratio: {ss4 / 2:.0f}×')

In [ ]:
# Visualise the combinatorial explosion
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Linear scale
ax1.bar(range(1, len(primes)+1), state_space_sizes, color='steelblue')
ax1.set_xlabel('Nesting depth (number of orbits)')
ax1.set_ylabel('Total state space size')
ax1.set_title('State Space Growth (linear scale)')
ax1.set_xticks(range(1, len(primes)+1))
ax1.set_xticklabels([f'{p}' for p in primes])

# Log scale
ax2.bar(range(1, len(primes)+1), np.log10(state_space_sizes), color='coral')
ax2.set_xlabel('Nesting depth (number of orbits)')
ax2.set_ylabel('log₁₀(state space size)')
ax2.set_title('State Space Growth (log scale)')
ax2.set_xticks(range(1, len(primes)+1))
ax2.set_xticklabels([f'{p}' for p in primes])

fig.suptitle('Combinatorial Explosion of Recurrence with Nesting Depth', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

### Approximate Recurrence: ε-returns

For a continuous system, we ask: how close does the outer state come to its initial state
over time? We measure the **minimum distance** achieved as a function of time horizon,
for progressively deeper nesting.

In [ ]:
# epsilon-recurrence: minimum state distance over time
four_primes = np.array([2, 3, 5, 7])
omega = 2 * np.pi * four_primes

# Fine time grid
t = np.linspace(0.001, 50, 500000)

# Compute angular positions
theta = np.array([np.mod(omega[i] * t, 2 * np.pi) for i in range(4)])

# For each nesting depth 1..4, compute circular distance of state from initial
initial = theta[:, 0]

fig, axes = plt.subplots(4, 1, figsize=(14, 12), sharex=True)
colors = ['#2196F3', '#4CAF50', '#FF9800', '#E91E63']

for depth in range(1, 5):
    # State = first 'depth' components
    diff = theta[:depth] - initial[:depth, np.newaxis]
    # Circular distance
    circ_diff = np.minimum(np.abs(diff), 2*np.pi - np.abs(diff))
    # L2 norm
    dist = np.sqrt(np.sum(circ_diff**2, axis=0))
    
    # Running minimum
    running_min = np.minimum.accumulate(dist)
    
    ax = axes[depth-1]
    ax.plot(t, running_min, color=colors[depth-1], linewidth=0.8)
    ax.set_ylabel(f'min dist (depth {depth})')
    ax.set_title(f'Depth {depth}: orbits {list(four_primes[:depth])}', loc='left', fontsize=10)
    ax.axhline(y=0.1, color='gray', linestyle='--', alpha=0.5, label='ε = 0.1')
    ax.legend(loc='upper right', fontsize=8)

axes[-1].set_xlabel('Time')
fig.suptitle('Running Minimum Distance from Initial State\n(lower = closer to recurrence)', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

### The Resolution Threshold

Count how many ε-returns occur within a fixed time window for each nesting depth.
The thesis predicts: inner orbits have many returns, outer orbits have progressively fewer,
creating a continuous gradient — not a binary split.

In [ ]:
# Count ε-returns for each depth at multiple ε values
epsilons = [0.5, 0.3, 0.1, 0.05, 0.01]
T_window = 50  # time window

print('Number of ε-returns within t ∈ [0, 50]:')
print('=' * 70)
header = f'{"Depth":>6} | {"Orbits":>12}'
for e in epsilons:
    header += f' | {f"ε={e}":>8}'
print(header)
print('-' * 70)

gradient_data = {}
for depth in range(1, 5):
    diff = theta[:depth] - initial[:depth, np.newaxis]
    circ_diff = np.minimum(np.abs(diff), 2*np.pi - np.abs(diff))
    dist = np.sqrt(np.sum(circ_diff**2, axis=0))
    
    row = f'{depth:>6} | {str(list(four_primes[:depth])):>12}'
    counts = []
    for e in epsilons:
        # Count passages below epsilon (with debouncing)
        below = dist < e
        transitions = np.diff(below.astype(int))
        n_returns = np.sum(transitions == 1)
        row += f' | {n_returns:>8}'
        counts.append(n_returns)
    print(row)
    gradient_data[depth] = counts

print('\n→ The gradient from returnable to non-returnable is continuous,')
print('  not a binary split. The resolution threshold is where returns')
print('  become too sparse to track within the observation window.')

In [ ]:
# Visualise the gradient
fig, ax = plt.subplots(figsize=(10, 6))

x = np.arange(1, 5)
width = 0.15
for i, e in enumerate(epsilons):
    counts = [gradient_data[d][i] for d in range(1, 5)]
    ax.bar(x + i*width, counts, width, label=f'ε = {e}', alpha=0.8)

ax.set_xlabel('Nesting depth')
ax.set_ylabel('Number of ε-returns in [0, 50]')
ax.set_title('Resolution Threshold:\nε-returns decrease continuously with nesting depth')
ax.set_xticks(x + 2*width)
ax.set_xticklabels([f'Depth {d}\n{list(four_primes[:d])}' for d in range(1, 5)])
ax.legend(title='Precision ε')
plt.tight_layout()
plt.show()

## Verdict

*(To be filled after running the computation)*